# Analyse de la qualité de l'air à Madagascar

Les données proviennent de l'API du dashboard (base PostgreSQL Neon).
Chaque bloc reproduit un graphique du dashboard.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

BASE = "https://aqi-std24015.vercel.app/api"

def get(path):
    return requests.get(f"{BASE}/{path}").json()

cities = pd.DataFrame(get("cities"))
timeseries = pd.DataFrame(get("timeseries"))
missing = pd.DataFrame(get("missing"))
pollutants = pd.DataFrame(get("pollutants"))
corr = get("correlations")
samples = pd.DataFrame(corr["samples"])

## AQI moyen par ville
Histogramme : comparaison directe entre les 5 villes.

In [ ]:
cities = cities.sort_values("avg_aqi", ascending=False)
plt.figure(figsize=(8, 4))
plt.bar(cities["city_name"], cities["avg_aqi"], color="#2dd4bf")
plt.title("AQI moyen par ville")
plt.ylabel("AQI")
plt.show()
print(cities[["city_name", "avg_aqi", "nb_mesures"]])

## Évolution journalière
Courbe : l'évolution dans le temps se lit de gauche à droite.

In [ ]:
pivot = timeseries.pivot(index="full_date", columns="city_name", values="avg_aqi")
pivot.plot(figsize=(10, 4))
plt.title("\u00c9volution journali\u00e8re de l'AQI")
plt.ylabel("AQI")
plt.show()

## Moyenne des polluants
Histogramme : les polluants les plus concentrés dans l'air.

In [ ]:
cols = ["pm2_5", "pm10", "no2", "o3"]
moy = pollutants[cols].mean()
plt.figure(figsize=(8, 4))
plt.bar(moy.index, moy.values, color=["#2dd4bf", "#fbbf24", "#fb7185", "#818cf8"])
plt.title("Moyenne des polluants")
plt.ylabel("\u00b5g/m\u00b3")
plt.show()

## Données manquantes
La base est complète : 0 % de données manquantes partout.

In [ ]:
print(missing[["city_name", "nb_mesures", "missing_pct"]])

## Relation PM2.5 vs AQI
Nuage de points : chaque point est une mesure. Plus les points s'alignent, plus la relation est forte.

In [ ]:
r = samples["pm2_5"].corr(samples["aqi"])
plt.figure(figsize=(8, 5))
plt.scatter(samples["pm2_5"], samples["aqi"], alpha=0.3, s=10)
plt.xlabel("PM2.5 (\u00b5g/m\u00b3)")
plt.ylabel("AQI")
plt.title(f"PM2.5 vs AQI - r = {r:.2f}")
plt.show()

## Matrice de corrélations
Toutes les relations entre polluants deux à deux.

In [ ]:
keys = ["pm2_5", "pm10", "no2", "o3", "co", "so2", "nh3", "aqi"]
M = samples[keys].corr().values
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(keys)), labels=keys)
ax.set_yticks(range(len(keys)), labels=keys)
for i in range(len(keys)):
    for j in range(len(keys)):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center")
plt.colorbar(im)
plt.title("Matrice de corr\u00e9lations")
plt.show()

## Conclusion
- Toliara est la ville la plus polluée, Antsiranana la plus saine.
- Les particules PM2.5 sont le principal facteur de l'AQI.
- La base contient 42 311 mesures, sans donnée manquante.